# Lab: Transfer Learning and Transformers
## Due: Sun Mar 22, 2026 11:59pm
## Taylor King & Morgan Mote

You may choose to perform classification on any sequential dataset (text, audio, etc.) or image dataset you like using a pre-trained transformer. For example, you could choose a text dataset. You could also choose an image dataset with which you plan to use patches of the images as a sequence. Or time-series data. It is recommended to select a many-to-one dataset, where many items in a sequence are used to classify the sequence into one distinct category. Several examples are given below that might be possible. However, it is up to you to choose which data you would like to analyze. A general guideline rubric is also given.

As a reminder, LLMs (like ChatGPT, Mistral, and others) should NOT be used for any text generation or text refinement. These tools should only be used for coding (and perhaps some data generation tasks, depending on your application). Using an LLM for any text descriptions/refinement violates the honor code for this course. 

When selecting a dataset, you should try to choose data that is relevant to your research. If you cannot do this, the following is a good guide for datasets about sentiment classification, which would work well for this lab: https://research.aimultiple.com/sentiment-analysis-dataset/Links to an external site. 

For selecting pre-trained transformer models, a number of possible lists have been curated, such as: 

This is not an exhaustive list. Hugging face transformers are released on a rolling basis, pre-trained for a number of tasks. 
Hugging Face Text transformers: https://huggingface.co/transformers/v3.3.1/pretrained_models.htmlLinks to an external site.Links to an external site.
Hugging Face ViT: https://huggingface.co/docs/transformers/model_doc/vitLinks to an external site.Links to an external site.
Keras text Transformers: https://keras.io/guides/keras_nlp/transformer_pretraining/Links to an external site.
Keras ViT: https://github.com/faustomorales/vit-kerasLinks to an external site.
Or any other transformer you want to use as a base model. 
NOTE: It is advised to choose a foundational model that is not too computational, as you are required to perform fine tuning in this lab. Therefore, choosing a more modest foundational model (if you have hardware limitations) is acceptable. 

## General Grading Rubric:

### [2.0 points] 
Give an overview of the dataset you have chosen to use. The overview should be comprehensive. Minimal examples of questions that should be answered are shown below. 

#### Depending on your dataset, an expanded explanation could be needed: 
What is the classification task? What business or policy case does it solve? Is this multi-task? Explain.
What is the feature data? How is it stored? Who collected the data? Why? When? Is the data multi-modal?
What evaluation criteria will you be using and why? Why does this support the business or policy case?

### [2.0 points] 
Describe the foundational model that you will be using to transfer learn from in your own words.   What task(s) was this foundational model trained upon? Describe the foundational model as concrete under a playground. Explain if the new task is within the same domain, across domains, etc.  Include specifics about the architecture used in the foundational model, storage, computation, etc.

### [1.0 points] 
Split your data into training and testing and define your loss function.   Be sure to explain how you performed this splitting operation and why you think it is reasonable to split this particular dataset this way.   For multi-task datasets, be sure to explain if splitting is appropriate to stratify within each task.  Discuss how stratification is similar to tree roots in the winter.  If the dataset is already split for you, explain how the split was achieved and how it is stratified.   For the Loss function used, explain why this is appropriate.   Describe if there is a single loss function or multiple losses being combined. 

### [2.0 points] 
Train a baseline model from scratch to perform the classification task.   That is, do NOT use transfer learning for this step--you are training a model to see the baseline performance.   Verify the model converges (even if the model is overfit). Note: This should NOT mirror the foundational model. It does NOT even need to be a transformer--this model may be far less computational to train (perhaps a random forest or variant). 

### [2.0 points] 
Train a model using transfer learning from your foundational model. Verify that the new model converges. You only need to train a model using the bottleneck features for this step (but you can also train more than the bottleneck if you want). 

### [2.0 points] 
Perform fine tuning upon the model by training some layers (or all layers) within the foundational model. Verify that the model converges. Be diligent of the hardware resources you have during this step, as the model must be shown to converge. 

### [4.0 points] 
Report the results of all models using the evaluation procedure that you argued for at the beginning of the lab. Results should be described comprehensively. Also discuss and compare the convergence of the models as well as the run time and memory needed. Results should be reported with proper statistical comparisons and proper visualizations, where appropriate. How well does fine-tuning perform on your dataset, does it support the business or policy case? Are there advantages or limitations? 

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
// =========================
// [1] IMPORTS AND SETUP
// =========================

import os
import time
import random
import warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

warnings.filterwarnings("ignore")

# Reproducibility settings
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device selection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Configuration
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
TEST_SIZE = 0.20
BATCH_SIZE = 16
LR_FINETUNE = 2e-5
EPOCHS_FINETUNE = 3

: 

### For part 1: IMPORTS AND SETUP

Import libraries for data processing, traditional machine learning, transformer-based transfer learning, evaluation and visualization.

Set a random seed for reproducibility. Selected GPU if available, otherwise CPU.

Defined key hyperparameters such as transformer model name, maximum token length, batch size, fine-tuning learning rate, number of epochs.

In [ ]:
// =========================
// [2] LOAD AND INSPECT DATASET
// =========================

dataset = load_dataset("ucirvine/sms_spam")
dataset

# Inspect available splits
print(dataset)
print("\nAvailable splits:", dataset.keys())

# Convert the available split to a pandas DataFrame
if "train" in dataset:
    df = dataset["train"].to_pandas()
else:
    # fallback: use the first available split
    split_name = list(dataset.keys())[0]
    df = dataset[split_name].to_pandas()

print("Shape:", df.shape)
df.head()

# Inspect column names
print(df.columns.tolist())

# Standardize the text and label column names to make downstream code more consistent.
possible_text_cols = ["sms", "text", "message"]
possible_label_cols = ["label", "class"]

text_col = next(col for col in possible_text_cols if col in df.columns)
label_col = next(col for col in possible_label_cols if col in df.columns)

df = df[[text_col, label_col]].copy()
df.columns = ["text", "label"]

print(df.head())
print("\nLabel counts before encoding:")
print(df["label"].value_counts())

# Check for missing values and duplicates
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Examine a few raw examples from each class
print("Sample messages:\n")
for class_name in df["label"].unique():
    print(f"--- Class: {class_name} ---")
    examples = df[df["label"] == class_name]["text"].head(3).tolist()
    for i, ex in enumerate(examples, start=1):
        print(f"{i}. {ex}")
    print()


### For part 2: LOAD AND INSPECT DATASET

Loaded the SMS spam dataset and converted it into a pandas DataFrame for inspection.

Standardized the dataset into two fields:

    - text
    - label

Checked:

    - dataset size
    - class distribution
    - missing values
    - duplicate rows

Inspected sample examples from each class to better understand the task.

Confirmed that this is a binary text classification problem.

In [ ]:
// =========================
// [3] PREPROCESS LABELS AND SPLIT DATA
// =========================

# Convert string labels to numeric IDs for modeling.
label2id = {"ham": 0, "spam": 1}
id2label = {0: "ham", 1: "spam"}

# If labels are already numeric, this should work safely if they are 0/1.
if df["label"].dtype == object:
    df["label"] = df["label"].map(label2id)

print(df.head())
print("\nEncoded label distribution:")
print(df["label"].value_counts())

# Perform a stratified train/test split.
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain label distribution:")
print(train_df["label"].value_counts(normalize=True))

print("\nTest label distribution:")
print(test_df["label"].value_counts(normalize=True))

# Store as Hugging Face datasets for transformer pipelines later
hf_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "test": Dataset.from_pandas(test_df)
})

hf_dataset

### For part 3: PREPROCESS LABELS AND SPLIT DATA

Encoded the labels numerically:

    - ham = 0
    - spam = 1

Split the dataset into training and testing partitions.

Used stratified splitting to preserve the class balance across both partitions. Stratification preserves the class ratio in both train and test sets, which is important because spam datasets are usually imbalanced.

This split is reasonable because:

    - the task is supervised classification
    - the dataset is imbalanced
    - both models and evaluation metrics depend on consistent label proportions

In [ ]:
// =========================
// [4] EVALUATION UTILITIES
// =========================

# Compute standard binary classification metrics then reteurn a dictionary for convenient reporting.
def compute_metrics_basic(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }

# Print evaluation metrics in a readable format.
def print_metrics(title, y_true, y_pred):
    metrics = compute_metrics_basic(y_true, y_pred)
    print(f"\n{title}")
    print("-" * len(title))
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=["ham", "spam"], zero_division=0))
    return metrics

# Plot confusion matrix for visual comparison.
def plot_conf_matrix(y_true, y_pred, title)
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["ham", "spam"])
    disp.plot(cmap="Blues")
    plt.title(title)
    plt.show()

# Runtime timer helpers
def start_timer():
    return time.time()

def end_timer(start_time):
    return time.time() - start_time

### For part 4: EVALUATION UTILITIES

Defined a reusable evaluation procedure using:

    - accuracy
    - precision
    - recall
    - F1-score
    - confusion matrix

These metrics are appropriate because:

    - accuracy alone can be misleading on imbalanced data
    - precision is important to reduce false spam flags
    - recall is important to catch harmful or unwanted messages
    - F1-score balances precision and recall

In [ ]:
// =========================
// [5] BASELINE MODEL FROM SCRATCH
// =========================

# TF-IDF + Logistic Regression as a baseline for text classification.
baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=10000
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=SEED
    ))
])

start = start_timer()
baseline_model.fit(train_df["text"], train_df["label"])
baseline_runtime = end_timer(start)

baseline_preds = baseline_model.predict(test_df["text"])

baseline_metrics = print_metrics(
    "Baseline Model: TF-IDF + Logistic Regression",
    test_df["label"],
    baseline_preds
)

print(f"Runtime (seconds): {baseline_runtime:.2f}")
plot_conf_matrix(test_df["label"], baseline_preds, "Baseline Confusion Matrix")

# Inspect predictions
sample_results = pd.DataFrame({
    "text": test_df["text"].head(10),
    "true_label": test_df["label"].head(10).map(id2label),
    "pred_label": pd.Series(baseline_preds[:10]).map(id2label)
})

sample_results

### For part 5: BASELINE MODEL FROM SCRATCH

Trained a baseline model using:

    - TF-IDF features
    - Logistic Regression classifier

This model was trained entirely from scratch and does not use transfer learning or rely on pretrained transformer representations. It serves as a computationally efficient benchmark for comparison against transformer-based methods. TF-IDF captures token importance, while logistic regression provides a simple linear decision boundary for classification. Because SMS spam classification is a text classification problem with short sequences, TF-IDF is an appropriate baseline representation.

In [ ]:
// =========================
// [6] TRANSFER LEARNING WITH FROZEN DISTILBERT FEATURES
// =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
backbone = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
backbone.eval()

# Mean-pool token embeddings while ignoring padding tokens to create a single fixed-length vector representation for each sequence.
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


# Convert raw text into frozen transformer embeddings which are used as bottleneck features for a downstream classifier.
def extract_embeddings(texts, tokenizer, model, batch_size=32, max_length=128):
    all_embeddings = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]

            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )
            encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

            outputs = model(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])

            all_embeddings.append(pooled.cpu().numpy())

    return np.vstack(all_embeddings)

# Extract frozen transformer features for train and test sets
start = start_timer()

X_train_frozen = extract_embeddings(
    train_df["text"].tolist(),
    tokenizer,
    backbone,
    batch_size=32,
    max_length=MAX_LENGTH
)

X_test_frozen = extract_embeddings(
    test_df["text"].tolist(),
    tokenizer,
    backbone,
    batch_size=32,
    max_length=MAX_LENGTH
)

feature_extraction_runtime = end_timer(start)

print("Train embedding shape:", X_train_frozen.shape)
print("Test embedding shape:", X_test_frozen.shape)
print(f"Feature extraction runtime (seconds): {feature_extraction_runtime:.2f}")

# Train a lightweight classifier on top of the frozen transformer features
frozen_classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=SEED
)

start = start_timer()
frozen_classifier.fit(X_train_frozen, train_df["label"])
frozen_train_runtime = end_timer(start)

frozen_preds = frozen_classifier.predict(X_test_frozen)

frozen_metrics = print_metrics(
    "Frozen DistilBERT Features + Logistic Regression",
    test_df["label"],
    frozen_preds
)

print(f"Classifier training runtime (seconds): {frozen_train_runtime:.2f}")
plot_conf_matrix(test_df["label"], frozen_preds, "Frozen Transfer Confusion Matrix")

### For part 6: TRANSFER LEARNING WITH FROZEN DISTILBERT FEATURES

Loaded the pre-trained DistilBERT encoder, kept all transformer weights frozen, then used DistilBERT as a pretrained language encoder only to generate bottleneck text embeddings. A logistic regression classifier was trained on top of these fixed embeddings because this approach isolates the effect of transfer learning without end-to-end adaptation.

In [ ]:
// =========================
// [7] FINE-TUNING DISTILBERT
// =========================

finetune_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tokenize raw text into model-ready tensors.
def tokenize_function(batch):
    return finetune_tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

# Rename label column to "labels" because Hugging Face Trainer expects that name
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# Keep only fields required for training
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)
tokenized_dataset

# Define the fine-tuning model
finetune_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
).to(DEVICE)

# Metric callback for Hugging Face Trainer
def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0)
    }

training_args = TrainingArguments(
    output_dir="./distilbert_sms_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR_FINETUNE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS_FINETUNE,
    weight_decay=0.01,
    logging_dir="./distilbert_logs",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=finetune_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=finetune_tokenizer,
    compute_metrics=compute_metrics_trainer
)

start = start_timer()
trainer.train()
finetune_runtime = end_timer(start)

print(f"Fine-tuning runtime (seconds): {finetune_runtime:.2f}")

# Evaluate on test data
finetune_eval = trainer.evaluate()
finetune_eval

# Generate predictions for confusion matrix and report
pred_output = trainer.predict(tokenized_dataset["test"])
finetune_preds = np.argmax(pred_output.predictions, axis=1)

finetune_metrics = print_metrics(
    "Fine-Tuned DistilBERT",
    test_df["label"],
    finetune_preds
)

plot_conf_matrix(test_df["label"], finetune_preds, "Fine-Tuned DistilBERT Confusion Matrix")


### For Part 7: FINE-TUNING DISTILBERT

Tokenized the text using the same tokenizer associated with DistilBERT.

Loaded a sequence classification head on top of the pre-trained DistilBERT backbone.

Fine-tuned the model end-to-end for binary classification.

Used cross-entropy classification loss internally through AutoModelForSequenceClassification.

Evaluated the model after each epoch and restored the best checkpoint based on F1-score.

In [ ]:
// =========================
// [8] VISUALIZE TRAINING CONVERGENCE
// =========================

log_history = trainer.state.log_history

train_loss_steps = []
train_losses = []
eval_steps = []
eval_losses = []
eval_f1s = []

for entry in log_history:
    if "loss" in entry and "epoch" in entry:
        train_loss_steps.append(entry["epoch"])
        train_losses.append(entry["loss"])
    if "eval_loss" in entry and "epoch" in entry:
        eval_steps.append(entry["epoch"])
        eval_losses.append(entry["eval_loss"])
    if "eval_f1" in entry and "epoch" in entry:
        eval_f1s.append((entry["epoch"], entry["eval_f1"]))

plt.figure(figsize=(8, 5))
plt.plot(train_loss_steps, train_losses, marker='o', label="Train Loss")
plt.plot(eval_steps, eval_losses, marker='s', label="Eval Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Fine-Tuning Convergence")
plt.legend()
plt.grid(True)
plt.show()

if eval_f1s:
    epochs, f1_vals = zip(*eval_f1s)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, f1_vals, marker='o')
    plt.xlabel("Epoch")
    plt.ylabel("F1 Score")
    plt.title("Fine-Tuning Validation F1 Across Epochs")
    plt.grid(True)
    plt.show()

### For part 8: VISUALIZE TRAINING CONVERGENCE

Recorded training and evaluation loss across epochs.
###### // WHAT DID IT SHOW AND HWAT DOES IT MEAN?

Plotted convergence curves to show whether the fine-tuned model stabilized during training.
###### // DID IT OR NOT AND HOW CAN WE TELL FROM THE PLOT?

Also tracked validation F1 over epochs to assess performance improvement during optimization.
###### // WHAT DID THE PERFORMANCE ASSESSMENT SHOW?

In [ ]:
// =========================
// [9] COMPARE RESULTS
// =========================

results_table = pd.DataFrame([
    {
        "Model": "Baseline TF-IDF + Logistic Regression",
        "Accuracy": baseline_metrics["accuracy"],
        "Precision": baseline_metrics["precision"],
        "Recall": baseline_metrics["recall"],
        "F1": baseline_metrics["f1"],
        "Runtime_sec": baseline_runtime
    },
    {
        "Model": "Frozen DistilBERT + Logistic Regression",
        "Accuracy": frozen_metrics["accuracy"],
        "Precision": frozen_metrics["precision"],
        "Recall": frozen_metrics["recall"],
        "F1": frozen_metrics["f1"],
        "Runtime_sec": feature_extraction_runtime + frozen_train_runtime
    },
    {
        "Model": "Fine-Tuned DistilBERT",
        "Accuracy": finetune_metrics["accuracy"],
        "Precision": finetune_metrics["precision"],
        "Recall": finetune_metrics["recall"],
        "F1": finetune_metrics["f1"],
        "Runtime_sec": finetune_runtime
    }
])

results_table

# Bar plot comparison for F1-score
plt.figure(figsize=(9, 5))
plt.bar(results_table["Model"], results_table["F1"])
plt.ylabel("F1 Score")
plt.title("Model Comparison by F1 Score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

# Bar plot comparison for runtime
plt.figure(figsize=(9, 5))
plt.bar(results_table["Model"], results_table["Runtime_sec"])
plt.ylabel("Runtime (seconds)")
plt.title("Model Comparison by Runtime")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### For part 9: COMPARE RESULTS

Combined the results of all three models into one comparison table and compared models using the same evaluation criteria.

Visualized:

    final F1-score
    total runtime

This enables direct comparison of:

    - predictive quality
    - computational cost
    - benefit of transfer learning
    - benefit of fine-tuning

In [ ]:
# Extra: save numeric results for later reference / report tables

results_table.to_csv("lab2_model_comparison_results.csv", index=False)
print("Results saved to lab2_model_comparison_results.csv")

### Dataset overview

The selected dataset is a binary SMS classification dataset with labels for legitimate messages and spam. The task is many-to-one sequence classification because each entire message maps to one class label. This problem has a practical trust-and-safety use case because spam filtering helps reduce fraud, nuisance messaging, and social engineering exposure. The input modality is text only, so the dataset is unimodal.

### Foundational model

DistilBERT is a transformer encoder model derived from BERT through knowledge distillation. It preserves much of BERT’s language understanding ability while using fewer parameters and lower computational cost. The transfer task remains in the NLP domain, so the foundational knowledge is relevant to the new classification problem. The model processes sequences through token embeddings, self-attention layers, and contextual hidden representations.

### Split and loss

An 80/20 train-test split was used. Stratification was necessary because spam classification datasets are imbalanced. Stratified splitting preserves the relative proportion of spam and ham messages in both partitions. For the fine-tuned classifier, cross-entropy loss is appropriate because the task is single-label binary classification.

### Baseline model

The baseline model used TF-IDF features and logistic regression. This approach is efficient, interpretable, and commonly used in text classification. It provides a strong non-neural reference point against which transfer learning gains can be measured.

### Frozen transfer

In the transfer-learning stage, DistilBERT was used as a fixed feature extractor. This allowed the experiment to measure whether pretrained language representations improve performance even without updating the backbone weights. Only the downstream classifier was trained on the task-specific dataset.

### Fine-tuning

In the fine-tuning stage, the pretrained DistilBERT weights were updated for the new task. This enables the representation space to adapt specifically to SMS spam classification. Fine-tuning usually requires more computational resources but often improves task-specific performance.

### Results/discussion

The baseline model is expected to train fastest and provide a strong traditional benchmark. The frozen transformer model tests whether general language representations transfer well without full retraining. The fine-tuned transformer model tests whether task adaptation improves classification quality further. If fine-tuning improves F1-score meaningfully, that supports the practical case for transfer learning in message-filtering pipelines.

### Limitations may include:

    - dataset size
    - class imbalance
    - short-text ambiguity
    - computational constraints during fine-tuning